# 03. Model performance (Table 2, Table 3, Figure 1)

* **C-index** (Harrell) and **IBS** for the training, validation and test sets (Table 2).
* **Time-dependent ROC/AUC** at 2, 5, 7 and 10 years (Figure 1): cumulative/dynamic definition with
  inverse probability of censoring weighting (IPCW, Uno). Cases: death at or before *t*; controls: alive
  beyond *t*; censoring distribution from the training set; score = predicted probability of death by *t*, 1 - S(t).
* **Point-specific metrics** (Table 3): deaths before *t* (positive) vs. patients censored at or after *t*
  (negative); threshold = Youden index on the validation set.
* Calibration at each time point.

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sksurv.util import Surv

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH    = Path("../data/dummy_data.csv")     # replace with your own data
OUTPUT_DIR   = Path("../outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
PREPROC_PATH = OUTPUT_DIR / "preprocessor.joblib"  # written by 01_data_preprocessor
MODEL_PATH   = OUTPUT_DIR / "RSF_model.joblib"     # written by 02_RSF_train
RANDOM_SEED  = 42

TIME_COL, EVENT_COL, ID_COLS = "OS", "event", ["SubjectKey"]   # OS in days

def load_splits(path=DATA_PATH):
    """Return {"Train"|"Val"|"Test": (X_raw, y)} using the column split_3."""
    df = pd.read_csv(path)
    df[EVENT_COL] = df[EVENT_COL].astype(bool)
    out = {}
    for name, key in [("Train", "train"), ("Val", "val"), ("Test", "test")]:
        d = df[df["split_3"] == key]
        out[name] = (d.drop(columns=ID_COLS + [TIME_COL, EVENT_COL]),
                     Surv.from_arrays(d[EVENT_COL], d[TIME_COL]))
    return out

In [ ]:
class Pred:
    """Predictions of a fitted RSF for one evaluated set."""
    def __init__(self, model, X, y):
        self.risk  = model.predict(X)
        self.S     = model.predict_survival_function(X, return_array=True)
        self.T     = model.unique_times_
        self.event = y["event"]
        self.time  = y["time"]

    def surv_at(self, rows, grid):
        """Step-function survival S(t) at the given times for the given rows."""
        grid = np.atleast_1d(np.asarray(grid, dtype=float))
        idx  = np.searchsorted(self.T, grid, side="right") - 1
        out  = np.ones((len(rows), len(grid)))
        ok   = idx >= 0
        out[:, ok] = self.S[rows][:, idx[ok]]
        return out


from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc, integrated_brier_score

TIME_POINTS = [730, 1825, 2555, 3650]                 # days
TP_LABELS   = ["2-year", "5-year", "7-year", "10-year"]

def metrics(pred, y_ref):
    """C-index (Harrell), IBS and 2/5/7/10-year TD-AUC; censoring distribution from y_ref (training set).
    IBS: 100-point grid from the minimum to the maximum follow-up of the evaluated set (end point excluded).
    TD-AUC: cumulative/dynamic AUC with IPCW (Uno); score = 1 - S(t)."""
    ev, tm = pred.event, pred.time
    y = Surv.from_arrays(ev, tm)
    rows = np.arange(len(tm))
    out = {"C-index": concordance_index_censored(ev, tm, pred.risk)[0]}
    grid = np.linspace(tm.min(), tm.max(), 100, endpoint=False)
    out["IBS"] = integrated_brier_score(y_ref, y, pred.surv_at(rows, grid), grid)
    p_death = 1.0 - pred.surv_at(rows, TIME_POINTS)
    for k, lbl in enumerate(TP_LABELS):
        out[f"{lbl} AUC"] = cumulative_dynamic_auc(y_ref, y, p_death[:, k], [TIME_POINTS[k]])[0][0]
    return out

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from sksurv.nonparametric import kaplan_meier_estimator

splits = load_splits()
y_train = splits["Train"][1]
preprocessor = joblib.load(PREPROC_PATH)   # from 01_data_preprocessor.ipynb
model        = joblib.load(MODEL_PATH)     # from 02_RSF_train.ipynb
X = {s: preprocessor.transform(splits[s][0]) for s in splits}
Y = {s: splits[s][1] for s in splits}
print({s: X[s].shape for s in X})

In [ ]:
# Table 2: C-index and IBS
PRED = {s: Pred(model, X[s], Y[s]) for s in ("Train", "Val", "Test")}
RES  = {s: metrics(PRED[s], y_train) for s in PRED}
print(pd.DataFrame(RES).T.round(4).to_string())

In [ ]:
# IPCW weights: censoring survival G(t-) estimated on the training set
_cens_t, _cens_s = kaplan_meier_estimator(~y_train["event"], y_train["time"])

def G(t):
    idx = int(np.searchsorted(_cens_t, t, side="left")) - 1
    return 1.0 if idx < 0 else max(float(_cens_s[idx]), 1e-8)

def ipcw_cases_controls(y_eval, tp):
    """Cases: event at or before tp (weight 1/G(T-)); controls: time > tp (weight 1/G(tp))."""
    t, e = y_eval["time"], y_eval["event"]
    idx_c = np.where((t <= tp) & e)[0]
    idx_k = np.where(t > tp)[0]
    idx     = np.concatenate([idx_c, idx_k])
    labels  = np.concatenate([np.ones(len(idx_c)), np.zeros(len(idx_k))])
    weights = np.concatenate([[1.0 / G(t[i]) for i in idx_c], np.full(len(idx_k), 1.0 / G(tp))])
    return idx, labels, weights

def td_roc(pred, y_eval, tp):
    """IPCW-weighted ROC at tp; its AUC equals sksurv cumulative_dynamic_auc."""
    p_death = 1.0 - pred.surv_at(np.arange(len(pred.time)), [tp])[:, 0]
    idx, labels, w = ipcw_cases_controls(y_eval, tp)
    fpr, tpr, _ = roc_curve(labels, p_death[idx], sample_weight=w)
    return fpr, tpr, roc_auc_score(labels, p_death[idx], sample_weight=w)

# consistency check against sksurv
for tp, lbl in zip(TIME_POINTS, TP_LABELS):
    print(f"{lbl}: IPCW ROC AUC = {td_roc(PRED['Test'], Y['Test'], tp)[2]:.4f} | "
          f"cumulative_dynamic_auc = {RES['Test'][lbl + ' AUC']:.4f}")

In [ ]:
# Figure 1: time-dependent ROC curves, test set
COLORS = dict(zip(TP_LABELS, ["tab:blue", "tab:orange", "tab:green", "tab:red"]))
fig, ax = plt.subplots(figsize=(5, 5))
for tp, lbl in zip(TIME_POINTS, TP_LABELS):
    fpr, tpr, auc = td_roc(PRED["Test"], Y["Test"], tp)
    grid = np.linspace(0, 1, 500)
    ax.plot(grid, np.interp(grid, fpr, tpr), color=COLORS[lbl], label=f"{lbl} AUC = {auc:.4f}")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.7)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("1 - Specificity"); ax.set_ylabel("Sensitivity")
ax.set_title("Time-dependent AUC (test set)")
ax.legend(fontsize=8, loc="lower right"); ax.grid(linestyle="--", alpha=0.3)
plt.tight_layout()
# plt.savefig("Figure1_td_roc_test.jpg", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Table 3: point-specific metrics with validation-derived Youden thresholds
def binary_at(pred, tp):
    t, e = pred.time, pred.event
    dead  = (t < tp) & e
    valid = dead | ((t >= tp) & ~e)
    p_death = 1.0 - pred.surv_at(np.arange(len(t)), [tp])[:, 0]
    return dead[valid].astype(int), p_death[valid]

THR = {}
for tp in TIME_POINTS:
    lab, score = binary_at(PRED["Val"], tp)
    fpr, tpr, thr = roc_curve(lab, score)
    THR[tp] = thr[np.argmax(tpr - fpr)]

rows = []
for tp, lbl in zip(TIME_POINTS, TP_LABELS):
    lab, score = binary_at(PRED["Test"], tp)
    pred = (score >= THR[tp]).astype(int)
    tn, fp, fn, tp_ = confusion_matrix(lab, pred, labels=[0, 1]).ravel()
    prec = tp_ / (tp_ + fp) if (tp_ + fp) else 0.0
    rec  = tp_ / (tp_ + fn) if (tp_ + fn) else 0.0
    rows.append({"Time point": lbl,
                 "Accuracy": (tp_ + tn) / len(lab),
                 "F1-score": 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0,
                 "Precision": prec, "Recall": rec,
                 "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
                 "Threshold": THR[tp], "n": len(lab)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
# Calibration at each time point (test set)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, tp, lbl in zip(axes, TIME_POINTS, TP_LABELS):
    lab, score = binary_at(PRED["Test"], tp)
    prob, obs = (1 - score) * 100, (1 - lab) * 100          # survival scale
    bins = np.linspace(0, 100, 81)
    idx  = np.digitize(prob, bins) - 1
    pts  = [(prob[idx == b].mean(), obs[idx == b].mean()) for b in range(80) if (idx == b).any()]
    ax.scatter(*zip(*pts), s=15)
    ax.plot([0, 100], [0, 100], "k--")
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.set_xlabel("Predicted survival (%)"); ax.set_ylabel("Observed survival (%)")
    ax.set_title(f"{lbl} calibration")
plt.tight_layout()
plt.show()